# E-Commerce Data Ingestion, Dynamic Upload & Column Mapping Pipeline

This notebook demonstrates how to:
1. **Load real-world Kaggle E-Commerce datasets** (UCI Online Retail or Brazilian Olist) directly with Pandas.
2. **Provide dynamic file upload** via `ipywidgets.FileUpload` (Jupyter) or `google.colab.files` (Colab).
3. **Execute an automated column-mapping step** immediately after loading to standardize schema for downstream EDA, RFM, and clustering.

## 1. Imports and Canonical Schema Definition

In [ ]:
import os
import io
import pandas as pd
import numpy as np

# Canonical Schema for downstream EDA, RFM, and Clustering
CANONICAL_COLUMNS = {
    'CustomerID': 'Unique identifier for customer',
    'InvoiceID': 'Unique identifier for order/transaction',
    'InvoiceDate': 'Date/timestamp of purchase',
    'Quantity': 'Item quantity purchased',
    'UnitPrice': 'Unit price of item',
    'TotalAmount': 'Total spend (Quantity * UnitPrice)',
    'Description': 'Item name or description',
    'Country': 'Customer country or store location'
}

# Common Kaggle presets and synonym mappings
PRESETS = {
    'uci_online_retail': {
        'CustomerID': 'CustomerID',
        'InvoiceNo': 'InvoiceID',
        'InvoiceDate': 'InvoiceDate',
        'Quantity': 'Quantity',
        'UnitPrice': 'UnitPrice',
        'Description': 'Description',
        'Country': 'Country'
    },
    'generic_synonyms': {
        'customer_id': 'CustomerID',
        'user_id': 'CustomerID',
        'client_id': 'CustomerID',
        'order_id': 'InvoiceID',
        'invoice_id': 'InvoiceID',
        'order_date': 'InvoiceDate',
        'invoice_date': 'InvoiceDate',
        'qty': 'Quantity',
        'price': 'UnitPrice',
        'unit_price': 'UnitPrice',
        'total': 'TotalAmount',
        'total_amount': 'TotalAmount',
        'total_spend': 'TotalAmount',
        'total_spend_usd': 'TotalAmount'
    }
}

## 2. Immediate Column-Mapping & Standardization Step

This function runs immediately after data is loaded to guarantee that downstream code never breaks.

In [ ]:
def standardize_columns(df: pd.DataFrame, preset='uci_online_retail', custom_mapping=None) -> pd.DataFrame:
    """
    Renames columns to canonical schema, computes TotalAmount if missing,
    parses dates, and validates RFM requirements.
    """
    df = df.copy()
    mapping = {}
    
    # 1. Preset dictionary
    if preset in PRESETS:
        for raw_col, target in PRESETS[preset].items():
            if raw_col in df.columns:
                mapping[raw_col] = target
                
    # 2. Synonym fuzzy match
    synonyms = PRESETS['generic_synonyms']
    for col in df.columns:
        normalized = col.strip().lower().replace(' ', '_').replace('-', '_')
        if normalized in synonyms and col not in mapping:
            mapping[col] = synonyms[normalized]
            
    # 3. Custom user override
    if custom_mapping:
        mapping.update(custom_mapping)
        
    # Apply rename
    print("--- Column Mapping Summary ---")
    for src, target in mapping.items():
        if src in df.columns:
            print(f"  Mapped: '{src}' -> '{target}'")
    df.rename(columns=mapping, inplace=True)
    
    # 4. Handle TotalAmount calculation
    if 'TotalAmount' not in df.columns and 'Quantity' in df.columns and 'UnitPrice' in df.columns:
        df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce').fillna(0)
        df['UnitPrice'] = pd.to_numeric(df['UnitPrice'], errors='coerce').fillna(0)
        df['TotalAmount'] = (df['Quantity'] * df['UnitPrice']).round(2)
        print("  [CALCULATED] 'TotalAmount' = Quantity * UnitPrice")
        
    # 5. Date parsing
    if 'InvoiceDate' in df.columns:
        df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
        print("  [PARSED] 'InvoiceDate' converted to datetime")
        
    # 6. Verification
    required = ['CustomerID', 'InvoiceDate', 'TotalAmount']
    missing = [c for c in required if c not in df.columns]
    if missing:
        print(f"\n[WARNING] Missing downstream RFM columns: {missing}")
    else:
        print("\n[VERIFIED] All core RFM fields present: CustomerID, InvoiceDate, TotalAmount")
        
    return df

## 3. Direct Loading: Kaggle UCI 'E-Commerce Data' (Online Retail)

If you downloaded `data.csv` from Kaggle (https://www.kaggle.com/datasets/carrie1/ecommerce-data):
- Use `encoding='ISO-8859-1'` or `'latin1'`.
- Drop null `CustomerID` values (guest checkouts).
- Filter out cancellations (negative quantities / invoice starting with 'C').

In [ ]:
def load_uci_kaggle_dataset(csv_path='data.csv') -> pd.DataFrame:
    # Load with latin1 encoding to handle European special characters
    raw_df = pd.read_csv(csv_path, encoding='ISO-8859-1', low_memory=False)
    print(f"Loaded {len(raw_df):,} rows from '{csv_path}'.")
    
    # Immediately apply column mapping
    df = standardize_columns(raw_df, preset='uci_online_retail')
    
    # Clean cancellations and guest checkouts
    df = df.dropna(subset=['CustomerID'])
    df['CustomerID'] = df['CustomerID'].astype(int).astype(str)
    
    # Filter out cancellations
    valid_sales = (~df['InvoiceID'].astype(str).str.startswith('C', na=False)) & (df['Quantity'] > 0)
    df = df[valid_sales].copy()
    
    # Re-calculate TotalAmount
    df['TotalAmount'] = (df['Quantity'] * df['UnitPrice']).round(2)
    print(f"Cleaned dataset contains {len(df):,} valid transactions.")
    return df

# Example usage if data.csv is present in workspace:
# df = load_uci_kaggle_dataset('data/raw/data.csv')

## 4. Dynamic File Upload Feature

### Option A: In Jupyter Notebook using `ipywidgets`
Run the cells below in Jupyter Notebook or VS Code Interactive window:

In [ ]:
import ipywidgets as widgets
from IPython.display import display

uploader = widgets.FileUpload(
    accept='.csv, .xlsx',
    multiple=False,
    description='Upload Dataset'
)
display(uploader)

In [ ]:
def parse_jupyter_upload(uploader):
    if not uploader.value:
        raise ValueError("Please select a file to upload first.")
    # Handle ipywidgets v7 and v8
    item = uploader.value[0] if isinstance(uploader.value, (list, tuple)) else next(iter(uploader.value.values()))
    content = item.get('content') if isinstance(item, dict) else item.content
    name = item.get('name') if isinstance(item, dict) else item.name
    
    print(f"Processing uploaded file: {name}")
    if name.endswith('.xlsx') or name.endswith('.xls'):
        raw_df = pd.read_excel(io.BytesIO(content))
    else:
        raw_df = pd.read_csv(io.BytesIO(content), encoding='ISO-8859-1', low_memory=False)
    
    # Immediately apply column mapping
    return standardize_columns(raw_df, preset='uci_online_retail')

# After uploading a file via the button above, run:
# df = parse_jupyter_upload(uploader)
# df.head()

### Option B: In Google Colab using `google.colab.files`

In [ ]:
def upload_in_colab():
    from google.colab import files
    print("Please select your custom CSV file to upload:")
    uploaded = files.upload()
    filename, content = next(iter(uploaded.items()))
    print(f"Uploaded {filename} ({len(content):,} bytes)")
    
    raw_df = pd.read_csv(io.BytesIO(content), encoding='ISO-8859-1', low_memory=False)
    return standardize_columns(raw_df, preset='uci_online_retail')

# Usage in Colab:
# df = upload_in_colab()
# df.head()

## 5. Downstream RFM Calculation (Recency, Frequency, Monetary)

Once standardized, the transactions can be aggregated into customer-level RFM features for K-Means clustering and predictive models.

In [ ]:
def compute_rfm(df: pd.DataFrame) -> pd.DataFrame:
    snapshot = df['InvoiceDate'].max() + pd.Timedelta(days=1)
    rfm = df.groupby('CustomerID').agg(
        Recency_Days=('InvoiceDate', lambda d: int((snapshot - d.max()).days)),
        Total_Purchases=('InvoiceID', 'nunique'),
        Total_Spend_USD=('TotalAmount', 'sum'),
        Avg_Order_Value_USD=('TotalAmount', 'mean')
    ).reset_index()
    
    rfm['Total_Spend_USD'] = rfm['Total_Spend_USD'].round(2)
    rfm['Avg_Order_Value_USD'] = rfm['Avg_Order_Value_USD'].round(2)
    return rfm

# Example:
# rfm_df = compute_rfm(df)
# rfm_df.to_csv('data/processed/cleaned_customer_data.csv', index=False)